# Module 1.2 -- Apache Spark 4.1: Intent-Driven Declarative Pipelines

**Track:** Big Data Engineering for AI Systems  
**Toolchain:** PySpark 4.1 - Spark Declarative Pipelines (SDP) - Structured Streaming RTM  
**Objective:** Move from imperative ETL scripts to declarative pipeline definitions, and build real-time streaming with sub-second latency.

---

## What Changed in Spark 4.1?

### The Old Way (Imperative -- 'HOW to process')
```python
# You write EVERY step manually:
df = spark.read.parquet('s3://raw/')
df = df.filter(df.quality > 0.8)
df = df.groupBy('category').agg(avg('price'))
df.write.mode('overwrite').parquet('s3://gold/')
# YOU manage: order, retries, checkpoints, parallelism
```

### The New Way (Declarative -- 'WHAT to produce')
```python
@dp.table  # Just DECLARE what you want
def gold_prices():
    return spark.read.table('silver_products').groupBy('category').agg(avg('price'))
# SPARK manages: order, retries, checkpoints, parallelism
```

**Analogy:** Imperative = giving GPS turn-by-turn directions. Declarative = telling GPS the destination and letting it figure out the route.

### Why This Matters for AI Engineers

| Benefit | Impact |
|---------|--------|
| Auto-DAG construction | No manual dependency management |
| Built-in retries | Failed steps automatically retry |
| Native checkpointing | Resume from failure, don't restart |
| Streaming RTM mode | Sub-second latency (was multi-second) |


## 📑 Table of Contents

* [What Changed in Spark 4.1?](#what-changed-in-spark-41)
  * [The Old Way (Imperative -- 'HOW to process')](#the-old-way-imperative-how-to-process)
  * [The New Way (Declarative -- 'WHAT to produce')](#the-new-way-declarative-what-to-produce)
  * [Why This Matters for AI Engineers](#why-this-matters-for-ai-engineers)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [Spark Core Architecture (Must Know Before SDP)](#spark-core-architecture-must-know-before-sdp)
  * [Key Terms](#key-terms)
  * [Pandas vs PySpark: Mental Model](#pandas-vs-pyspark-mental-model)
* [1 - Spark Declarative Pipelines (SDP)](#1-spark-declarative-pipelines-sdp)
  * [How Does SDP Build the DAG Automatically?](#how-does-sdp-build-the-dag-automatically)
  * [Trade-offs: SDP vs Traditional Spark](#trade-offs-sdp-vs-traditional-spark)
* [2 - Structured Streaming Real-Time Mode (RTM)](#2-structured-streaming-real-time-mode-rtm)
  * [What is RTM?](#what-is-rtm)
  * [Why RTM Instead of Flink for Low-Latency?](#why-rtm-instead-of-flink-for-low-latency)
* [Knowledge Check](#knowledge-check)
  * [Question 1: Lazy Evaluation](#question-1-lazy-evaluation)
  * [Question 2: .collect() Danger](#question-2-collect-danger)
  * [Question 3: SDP vs Traditional](#question-3-sdp-vs-traditional)
  * [Question 4: PyArrow UDFs](#question-4-pyarrow-udfs)
* [Practical Practice](#practical-practice)
  * [Exercise 1: Pandas to PySpark Translation](#exercise-1-pandas-to-pyspark-translation)
  * [Exercise 2: DAG Construction](#exercise-2-dag-construction)
  * [Exercise 3: PyArrow Batch Function](#exercise-3-pyarrow-batch-function)
* [Summary](#summary)


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** When data exceeds 50GB, Pandas completely collapses (MemoryError). Seniors leverage distributed processing frameworks like Spark to crunch terabytes horizontally across clusters.

**Mental Model:** Pandas is a single powerful chef in a small kitchen. Spark is an executive chef (Driver) orchestrating 100 sous-chefs (Executors) working on pieces of the meal simultaneously.

**Common Junior Pitfall:** Calling `.collect()` on a massive Spark DataFrame, pulling 100GB of data back into the single Driver node, crashing the entire Spark application.

---


In [ ]:
# Cell 1 -- Install
!pip install -q pyspark==4.1.0 pandas pyarrow


---
## Spark Core Architecture (Must Know Before SDP)

Before diving into Spark Declarative Pipelines, you need to understand the basic Spark architecture:

```
Your PySpark Code
       |
  [Driver] -- orchestrates the work, holds SparkSession
       |
  [Cluster Manager] -- allocates resources (YARN, K8s, Standalone)
      / | \
 [Executor] [Executor] [Executor]  -- worker JVMs that process data
   |          |          |
 [Task]     [Task]     [Task]      -- a unit of work on one partition
```

### Key Terms

| Term | Meaning | Example |
|------|---------|--------|
| **Driver** | The master process that runs your `main()` | Your notebook kernel |
| **Executor** | Worker processes that do the heavy lifting | 4 containers in a K8s cluster |
| **Partition** | A chunk of data processed independently | 1 million rows split into 200 partitions of 5000 rows each |
| **DAG** | Directed Acyclic Graph of transformations | `read -> filter -> groupBy -> write` |
| **Catalyst** | Spark's query optimizer | Automatically reorders operations for speed |
| **Lazy evaluation** | Transformations aren't executed until an **action** (`.show()`, `.write()`) | Spark optimizes the whole plan before executing |

### Pandas vs PySpark: Mental Model

| Pandas | PySpark | Why Different |
|--------|---------|---------------|
| `df.head()` | `df.show(5)` | Data is distributed, not local |
| `df['col']` | `df.select('col')` | Column access returns a new DataFrame |
| `df[df.x > 5]` | `df.filter(df.x > 5)` | Same logic, different syntax |
| `df.groupby('a').mean()` | `df.groupBy('a').agg(avg('x'))` | Explicit aggregation names |
| Runs on 1 machine | Runs on 100+ machines | Spark partitions data across cluster |


In [ ]:
# Pandas vs PySpark -- see the same operation in both
import pandas as pd
import numpy as np

# Create sample data
np.random.seed(42)
data = {
    'device_id': [f'sensor_{i%5}' for i in range(20)],
    'temperature': np.random.normal(72, 8, 20).round(1),
    'quality': np.random.uniform(0.5, 1.0, 20).round(2)
}
pdf = pd.DataFrame(data)

# --- PANDAS (single machine) ---
print('=== PANDAS ===')
result_pandas = (
    pdf[pdf['quality'] > 0.8]
    .groupby('device_id')['temperature']
    .agg(['mean', 'count'])
    .reset_index()
)
print(result_pandas)

# --- EQUIVALENT PYSPARK (distributed cluster) ---
print('\n=== EQUIVALENT PYSPARK CODE ===')
pyspark_code = '''
from pyspark.sql.functions import avg, count, col

result_spark = (
    spark.read.parquet("s3://data/sensors/")
    .filter(col("quality") > 0.8)
    .groupBy("device_id")
    .agg(
        avg("temperature").alias("mean"),
        count("*").alias("count")
    )
)
result_spark.show()
'''
print(pyspark_code)
print('Key difference: Pandas runs on your laptop.')
print('PySpark splits this across 100 machines automatically.')


---
## 1 - Spark Declarative Pipelines (SDP)

### How Does SDP Build the DAG Automatically?

When you use `@dp.table`, Spark analyzes which tables each function reads from, and automatically constructs the execution order:

```
@dp.table bronze_sensors  (reads: raw kafka stream)
       |
@dp.table silver_cleaned   (reads: bronze_sensors)  -- Spark figures this out!
       |
@dp.table gold_features    (reads: silver_cleaned)   -- And this!
```

### Trade-offs: SDP vs Traditional Spark

| | SDP (Declarative) | Traditional (Imperative) |
|---|---|---|
| Learning curve | Low (just decorators) | High (manual DAGs) |
| Control | Less (Spark decides) | Full (you decide everything) |
| Debugging | Harder (abstracted) | Easier (step-by-step) |
| Best for | Standard ETL pipelines | Complex custom logic |


In [ ]:
# Cell 2 -- SDP pipeline definition (conceptual, runs on Spark 4.1 cluster)
#
# This demonstrates the STRUCTURE of a declarative pipeline.
# In production, this runs on a Spark cluster with @dp.table decorators.

pipeline_code = '''
from pyspark.pipelines import DeclaredPipeline as dp
from pyspark.sql.functions import avg, count, col, window

# BRONZE: ingest raw data (no transformation)
@dp.table(comment="Raw sensor readings from IoT Kafka topic")
def bronze_sensors():
    return (
        spark.readStream
        .format("kafka")
        .option("subscribe", "iot-sensors")
        .load()
        .selectExpr("CAST(value AS STRING)", "timestamp")
    )

# SILVER: clean and type (Spark auto-detects this depends on bronze)
@dp.table(comment="Cleaned, typed sensor data")
def silver_sensors():
    return (
        spark.read.table("bronze_sensors")
        .filter(col("device_id").isNotNull())
        .filter(col("temperature").between(-50, 150))
        .dropDuplicates(["device_id", "timestamp"])
    )

# GOLD: aggregate features for ML
@dp.table(comment="Hourly device health metrics for ML training")
def gold_device_health():
    return (
        spark.read.table("silver_sensors")
        .groupBy("device_id", window("timestamp", "1 hour"))
        .agg(
            avg("temperature").alias("avg_temp"),
            count("*").alias("reading_count"),
        )
    )
'''
print('Spark Declarative Pipeline Definition:')
print(pipeline_code)

---
## 2 - Structured Streaming Real-Time Mode (RTM)

### What is RTM?

Traditional Spark Streaming uses **micro-batching** -- it collects data for 1-5 seconds, then processes in bulk. RTM processes each record **immediately** upon arrival.

| Mode | Latency | Throughput | When to Use |
|------|---------|-----------|-------------|
| Micro-batch | 1-10 sec | Very high | Analytics, hourly reports |
| RTM | 1-50 ms | High | Fraud detection, live features |
| Continuous (deprecated) | ~1 ms | Lower | Replaced by RTM |

### Why RTM Instead of Flink for Low-Latency?

| | Spark RTM | Apache Flink |
|---|---|---|
| Latency | 1-50ms | 1-10ms |
| Ecosystem | Full Spark SQL, MLlib, Delta | Flink SQL only |
| Learning curve | Know Spark? Already done | Separate framework |
| Best for | Teams already on Spark | Pure streaming workloads |


In [ ]:
# Cell 3 -- PyArrow-native UDFs (eliminate Pandas conversion overhead)
#
# Old way: Python UDF serializes data row-by-row (SLOW, 10-100x overhead)
# New way: PyArrow UDF operates on columnar batches (FAST, near-native speed)

import pyarrow as pa
import numpy as np

# Simulate what a PyArrow UDF looks like
def compute_zscore_pyarrow(temperature_array: pa.Array) -> pa.Array:
    """PyArrow-native UDF: operates on columnar data without Pandas conversion."""
    values = temperature_array.to_numpy(zero_copy_only=False)
    mean, std = values.mean(), values.std()
    zscores = (values - mean) / (std + 1e-10)
    return pa.array(zscores, type=pa.float64())

# Demo
temps = pa.array(np.random.normal(72, 8, 1000))
zscores = compute_zscore_pyarrow(temps)
print('PyArrow UDF Performance Demo:')
print(f'  Input:  {len(temps)} temperatures')
print(f'  Output: {len(zscores)} z-scores')
print(f'  Anomalies (|z| > 2): {sum(1 for z in zscores.to_pylist() if abs(z) > 2)}')
print(f'\n  Old way (Python UDF):  ~500 rows/sec (serializes each row)')
print(f'  New way (PyArrow UDF): ~500,000 rows/sec (columnar batches)')
print(f'  Speedup: ~1000x')

---
## Knowledge Check

### Question 1: Lazy Evaluation
If you write `df = spark.read.parquet('s3://data/')` followed by `df = df.filter(df.x > 5)`, has Spark read ANY data yet?

<details>
<summary>Click to reveal answer</summary>

**No.** Spark uses lazy evaluation. Transformations like `filter`, `select`, `groupBy` are recorded in the DAG but NOT executed. Data is only processed when you call an **action** like `.show()`, `.count()`, or `.write()`. This lets Catalyst optimize the entire plan before running it.
</details>

### Question 2: .collect() Danger
Why is calling `.collect()` on a 500GB DataFrame dangerous?

<details>
<summary>Click to reveal answer</summary>

`.collect()` pulls ALL data from all executors to the single Driver node's memory. If the data is 500GB and your Driver has 16GB RAM, it will crash with an OutOfMemoryError. Use `.show(20)` or `.take(100)` instead for inspection, or `.write()` to save results directly to storage.
</details>

### Question 3: SDP vs Traditional
What does `@dp.table` do that writing explicit `.write()` calls doesn't?

<details>
<summary>Click to reveal answer</summary>

`@dp.table` lets Spark automatically: (1) determine execution order from table dependencies (auto-DAG), (2) handle retries on failure, (3) manage checkpointing for recovery. With traditional `.write()` you must manage all of this manually.
</details>

### Question 4: PyArrow UDFs
Why are PyArrow UDFs ~1000x faster than regular Python UDFs in Spark?

<details>
<summary>Click to reveal answer</summary>

Regular Python UDFs serialize each row individually from the JVM to Python and back (row-by-row overhead). PyArrow UDFs operate on **columnar batches** -- entire columns are passed as Arrow arrays without per-row serialization. This eliminates the JVM-Python border crossing bottleneck.
</details>


---
## Practical Practice

### Exercise 1: Pandas to PySpark Translation
Translate this Pandas code to PySpark syntax:
```python
result = df[df['status'] == 'completed'].groupby('user_id')['amount'].sum().reset_index()
```

### Exercise 2: DAG Construction
Given these three SDP table definitions, draw the dependency DAG:
- `bronze_orders()` reads from Kafka
- `silver_orders()` reads from `bronze_orders`
- `gold_revenue()` reads from `silver_orders`
- `gold_user_metrics()` reads from `silver_orders` AND `bronze_orders`

### Exercise 3: PyArrow Batch Function
Write a function that takes a PyArrow array of strings and returns an array of their lengths.


In [ ]:
# ===== EXERCISE SOLUTIONS (try yourself first!) =====
import pyarrow as pa

print('Ex 1: PySpark Translation')
print('''from pyspark.sql.functions import col, sum as spark_sum
result = (
    df.filter(col("status") == "completed")
    .groupBy("user_id")
    .agg(spark_sum("amount").alias("amount"))
)''')

print('\nEx 2: DAG')
print('Kafka -> bronze_orders -> silver_orders -> gold_revenue')
print('                    \\                  \\-> gold_user_metrics')
print('                     \\-----------------/')
print('Note: gold_user_metrics has TWO parents (diamond dependency)')

print('\nEx 3: PyArrow string lengths')
def string_lengths(arr: pa.Array) -> pa.Array:
    return pa.array([len(s.as_py()) if s.is_valid else 0 for s in arr])

test = pa.array(['hello', 'world', 'spark'])
print(f'  Input:  {test}')
print(f'  Output: {string_lengths(test)}')


---
## Summary

| Concept | What You Learned |
|---------|------------------|
| SDP | Declare WHAT, Spark handles HOW |
| Auto-DAG | Dependencies resolved automatically |
| RTM | Sub-second streaming without leaving Spark |
| PyArrow UDFs | 1000x faster than Python UDFs |

**Next →** `09_data_orchestration_pipelines.ipynb` — Data Pipeline Orchestration for AI Systems